# 🏭 Titan Maintenance Intelligence Platform (MIP)
### Agentic AI — Challenge 1 · Asset Reliability & Predictive Maintenance

This notebook runs the **complete 5-agent crew** end-to-end in Google Colab. It unites the three project files:

| File | Role |
|------|------|
| `step1_setup.py` | Environment + Gemini LLM setup |
| `step2_mock_tools.py` | The 12 mock tools the agents call |
| `titan_agents.py` | The 5 CrewAI agents + the `run_pipeline()` engine |

**Agents:** ⚙️ Orchestrator · 📡 Reliability · 🔍 Process · 💰 Cost · 🛡️ Safety & Governance

> Run the cells top to bottom. You'll be asked to paste your Gemini API key (it is **not** stored in the notebook).


## 1 — Get the project files into Colab
This clones the team repo so `step2_mock_tools.py` and `titan_agents.py` are available. (If you already uploaded the files manually, you can skip this cell.)

In [ ]:
# If running in Colab, clone the repo. If the files are already here, this is a no-op.
import os
if not os.path.exists("titan_agents.py"):
    !git clone https://github.com/glorichin/Agentic.git
    %cd Agentic
print("Files present:", sorted(f for f in os.listdir() if f.endswith(('.py'))))

## 2 — Install dependencies
Wait for the green checkmark. Ignore the dependency warnings.

In [ ]:
!pip install -r requirements.txt -q

## 3 — Set your Gemini API key
Paste the team key when prompted. Using `getpass` keeps the key out of the saved notebook (so it never lands in GitHub). Get a free key at https://aistudio.google.com/apikey

In [ ]:
import os, getpass
os.environ["GEMINI_API_KEY"] = getpass.getpass("Paste your Gemini API key: ").strip()
os.environ.setdefault("GEMINI_MODEL", "gemini/gemini-2.0-flash")
print("Key set:", os.environ["GEMINI_API_KEY"][:6] + "…")

## 4 — Verify the tool layer (Step 2)
Quick smoke test of all 12 mock tools so we know the agents have good data to reason over. You should see `All 12 tools passed`.

In [ ]:
!python3 step2_mock_tools.py

## 5 — Run the crew
`run_pipeline()` builds the 5 agents, runs them as a sequential pipeline (specialists → governance gate → orchestrator synthesis), and returns every agent's output plus the final decision.

In [ ]:
from titan_agents import run_pipeline, DEMO_EVENT
import json

print("Trigger event:")
print(json.dumps(DEMO_EVENT, indent=2))
print("\nRunning the crew (this takes ~30–90s)…\n")

result = run_pipeline(DEMO_EVENT)

## 6 — Each agent's assessment

In [ ]:
import json
for key in ["reliability", "process", "cost", "governance"]:
    print("="*64)
    print(key.upper())
    print("="*64)
    a = result["agents"].get(key, {})
    print(json.dumps(a.get("parsed"), indent=2) if a.get("parsed") else a.get("raw"))
    print()

## 7 — Orchestrator's final decision

In [ ]:
import json
print("="*64)
print("ORCHESTRATOR — FINAL DECISION")
print("="*64)
print(json.dumps(result["final"], indent=2) if result["final"] else result["final_raw"])

## ✅ What just happened
The Orchestrator received the sensor breach, the **Reliability**, **Process** and **Cost** specialists each assessed it with their own tools, the **Safety & Governance** agent gated the proposed actions, and the Orchestrator checked for conflicts, computed the priority score, and returned a single **AUTONOMOUS / ESCALATE** decision with a rationale.

**Want the visual demo?** Run the Streamlit app locally:
```bash
pip install -r requirements.txt
streamlit run app.py
```
Change the asset / reading in `DEMO_EVENT` (or the Streamlit sidebar) to explore the LOW-risk autonomous path vs the CRITICAL escalation.
